# AMEX Enterprise Credit Risk Platform
## Notebook 32 — Phase 2, Problem 3: Expected Credit Loss (IFRS9/CECL) — Statistical Validation, Sensitivity & Deployment
### Problem Statement 3 of 14 (notebook 3 of 4)

CRISP-DM stage: **Evaluation / Deployment**. Depends on Notebook 31's real `ecl_by_customer.csv`.

**What this notebook does:** statistically validates that IFRS9 staging rank-orders with real outcomes (chi-square, two-proportion z-test, bootstrap CIs, split-half PSI — the same real tests Notebook 28 ran for Problem 4); runs a macro-scenario sensitivity analysis showing the real ECL range across Upside/Baseline/Downside; then packages the ECL *formula* layer (staging rubric, tier-LGD lookup, macro blend, discounting) into a standalone, deployable `ecl_calculator.py` — proven to reproduce Notebook 31's own real output on **every** holdout customer before deployment is declared ready.

**What this notebook does NOT try to reproduce:** Problem 1's actual PD model. That model is already Problem 1's own deployable artifact (`preprocessing_artifacts.joblib` + the champion model `.joblib` file) — re-bundling it here would be pure duplication. `ecl_calculator.py` takes a PD (however it was produced) and a severity tier as input and returns the ECL — it composes with Problem 1's real deployed model rather than re-implementing it.

**Lessons applied from Problem 4's real-run debugging (see project notes):** every numeric value the standalone calculator uses is read directly from Notebook 30's `ecl_policy.json`, never re-derived or rounded; the self-test checks **every** real holdout customer, not a sample; and any tier/stage mismatch is classified as a genuine defect only if the underlying dollar difference is non-trivial — a sub-cent difference at a stage boundary is expected floating-point noise between two independently-computed code paths, not a bug, and is reported as such rather than silently hidden.

**Deliverables:** `p3_statistical_validation.csv`, `p3_macro_sensitivity.csv`, `ecl_scoring_bundle.json`, `ecl_calculator.py`, `p3_deployment_readiness_checklist.csv`, 1 chart, `notebook_32_summary.json`.

**Run the single code cell below, once.** Idempotent — every output file is overwritten in place on every re-run.

In [ ]:
# =============================================================================
# SECTION 1: ENVIRONMENT SETUP -- LOAD NOTEBOOK 30/31's REAL RESULTS
# =============================================================================
import os
import sys
import json
import math
import warnings
import importlib.util
from pathlib import Path
from datetime import datetime, timezone


def _section(title: str) -> None:
    bar = "=" * 78
    print(f"\n{bar}\n{title}\n{bar}")


_section("SECTION 1: Environment Setup -- Load Notebook 30/31's Real Results")

PROJECT_ROOT = Path(r"C:\Users\rnand\Downloads\amex-default-prediction\AMEX_Enterprise_Credit_Risk_Platform")
P1_ARTIFACTS = PROJECT_ROOT / "Phase1_Foundation" / "Problem1_Credit_Scoring_PD_Prediction" / "artifacts"
P3_ROOT = PROJECT_ROOT / "Phase2_Regulatory_Loss_Provisioning" / "Problem3_Expected_Credit_Loss_IFRS9_CECL"
ARTIFACTS_DIR = P3_ROOT / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

PILLAR_DIRS = {
    "p3_policy": P3_ROOT / "01_ECL_Policy",
    "p3_modeling": P3_ROOT / "02_ECL_Modeling",
    "p3_validation_deployment": P3_ROOT / "03_Validation_Deployment",
    "p3_reporting_packaging": P3_ROOT / "04_Financial_Impact_Reporting_Packaging",
}
for _d in PILLAR_DIRS.values():
    _d.mkdir(parents=True, exist_ok=True)

P1_CONFIG_PATH = P1_ARTIFACTS / "project_config.json"
ECL_POLICY_PATH = PILLAR_DIRS["p3_policy"] / "ecl_policy.json"
ECL_BY_CUSTOMER_PATH = PILLAR_DIRS["p3_modeling"] / "ecl_by_customer.csv"
STAGE_SUMMARY_PATH = PILLAR_DIRS["p3_modeling"] / "ecl_stage_summary.csv"
for _p, _fix in [
    (P1_CONFIG_PATH, "run Problem 1's Notebook 01 first."),
    (ECL_POLICY_PATH, "run Notebook 30 first."),
    (ECL_BY_CUSTOMER_PATH, "run Notebook 31 first."),
    (STAGE_SUMMARY_PATH, "run Notebook 31 first."),
]:
    if not _p.exists():
        raise FileNotFoundError(f"{_p} not found.\nFix: {_fix}")

with open(P1_CONFIG_PATH, "r", encoding="utf-8") as f:
    P1_CONFIG = json.load(f)
with open(ECL_POLICY_PATH, "r", encoding="utf-8") as f:
    ECL_POLICY = json.load(f)

RANDOM_SEED = P1_CONFIG["random_seed"]
TIER_ORDER = ECL_POLICY["lgd_by_tier"]["tier_order"]
LGD_BY_TIER = ECL_POLICY["lgd_by_tier"]["values"]
EAD = ECL_POLICY["ead_per_account_usd"]
SICR_PD_MULTIPLE = ECL_POLICY["staging_criteria"]["sicr_pd_multiple"]
STAGE3_PD_THRESHOLD = ECL_POLICY["staging_criteria"]["stage3_pd_threshold"]
LIFETIME_PD_MULTIPLIER = ECL_POLICY["lifetime_pd"]["lifetime_pd_multiplier"]
MACRO_SCENARIOS = ECL_POLICY["macro_overlay"]["scenarios"]
ANNUAL_RATE = ECL_POLICY["discount_rate"]["annual_rate"]
STAGE1_DISCOUNT_YEARS = ECL_POLICY["discount_rate"]["stage1_discount_period_years"]
STAGE23_DISCOUNT_YEARS = ECL_POLICY["discount_rate"]["stage23_avg_remaining_life_years"]

print(f"Tier order (real, from Problem 4)   : {TIER_ORDER}")
print(f"LGD by tier (real, from Problem 4)  : {LGD_BY_TIER}")
print(f"Macro scenarios (ASSUMPTION)         : {[s['scenario'] for s in MACRO_SCENARIOS]}")
print("\n\u2705 Section 1 complete.")


# =============================================================================
# SECTION 2: LIBRARY IMPORTS
# =============================================================================
_section("SECTION 2: Library Imports")

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

missing = []
try:
    import numpy as np
except ImportError:
    missing.append("numpy")
try:
    import pandas as pd
except ImportError:
    missing.append("pandas")
try:
    from scipy.stats import chi2_contingency, norm
except ImportError:
    missing.append("scipy")
try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
except ImportError:
    missing.append("matplotlib")
if missing:
    raise ImportError("Missing required package(s): " + ", ".join(missing) +
                       "\nFix: pip install " + " ".join(missing))

np.random.seed(RANDOM_SEED)
print("\n\u2705 Section 2 complete.")


# =============================================================================
# SECTION 3: LOAD NOTEBOOK 31's REAL OUTPUTS
# =============================================================================
_section("SECTION 3: Load Notebook 31's Real Outputs")

ecl_df = pd.read_csv(ECL_BY_CUSTOMER_PATH)
ecl_df["severity_tier"] = pd.Categorical(ecl_df["severity_tier"], categories=TIER_ORDER, ordered=True)
print(f"Holdout customers loaded (real, from Notebook 31): {len(ecl_df):,}")
print(ecl_df["ifrs9_stage"].value_counts().sort_index().to_string())
print("\n\u2705 Section 3 complete.")


# =============================================================================
# SECTION 4: STATISTICAL VALIDATION -- CHI-SQUARE INDEPENDENCE + TWO-PROPORTION Z-TEST
# =============================================================================
_section("SECTION 4: Statistical Validation -- Chi-Square Independence + Two-Proportion Z-Test")

_contingency = pd.crosstab(ecl_df["ifrs9_stage"], ecl_df["target"])
_chi2, _chi2_p, _dof, _ = chi2_contingency(_contingency)
_n = len(ecl_df)
_cramers_v = float(np.sqrt(_chi2 / (_n * (min(_contingency.shape) - 1))))
print(f"Chi-square test (stage independent of default?): chi2={_chi2:.2f}, dof={_dof}, p-value={_chi2_p:.2e}")
print(f"Cram\u00e9r's V (effect size)                      : {_cramers_v:.4f}")

_stage1_row = _contingency.loc[1]
_stage3_row = _contingency.loc[3] if 3 in _contingency.index else None
if _stage3_row is None:
    raise RuntimeError("No Stage 3 customers in this holdout -- cannot run the Stage 3 vs Stage 1 z-test. "
                        "Fix: revisit Notebook 30's stage3_pd_threshold if this persists on real data.")
_n1, _x1 = int(_stage1_row.sum()), int(_stage1_row.get(1, 0))
_n3, _x3 = int(_stage3_row.sum()), int(_stage3_row.get(1, 0))
_p1, _p3 = _x1 / _n1, _x3 / _n3
_p_pool = (_x1 + _x3) / (_n1 + _n3)
_se = np.sqrt(_p_pool * (1 - _p_pool) * (1 / _n1 + 1 / _n3))
_z = (_p3 - _p1) / _se if _se > 0 else float("inf")
_z_p = float(2 * (1 - norm.cdf(abs(_z))))
print(f"Two-proportion z-test (Stage 3 vs Stage 1 default rate): z={_z:.2f}, p-value={_z_p:.2e} "
      f"(Stage1={_p1:.2%}, Stage3={_p3:.2%})")
print("\n\u2705 Section 4 complete.")


# =============================================================================
# SECTION 5: BOOTSTRAP CONFIDENCE INTERVALS ON PER-STAGE DEFAULT RATE
# =============================================================================
_section("SECTION 5: Bootstrap Confidence Intervals on Per-Stage Default Rate")

N_BOOT = 1000
_rng = np.random.default_rng(RANDOM_SEED)
_bootstrap_rows = []
for _s in sorted(ecl_df["ifrs9_stage"].unique()):
    _y = ecl_df.loc[ecl_df["ifrs9_stage"] == _s, "target"].to_numpy()
    _boot_means = np.array([_rng.choice(_y, size=len(_y), replace=True).mean() for _ in range(N_BOOT)])
    _lo, _hi = np.percentile(_boot_means, [2.5, 97.5])
    _bootstrap_rows.append({"ifrs9_stage": int(_s), "n": len(_y), "observed_default_rate": float(_y.mean()),
                             "ci_95_low": float(_lo), "ci_95_high": float(_hi)})
bootstrap_df = pd.DataFrame(_bootstrap_rows)
print(bootstrap_df.to_string(index=False))
print(f"\n(N_BOOT={N_BOOT} resamples per stage, random_seed={RANDOM_SEED} -- reproducible.)")
print("\n\u2705 Section 5 complete.")


# =============================================================================
# SECTION 6: SPLIT-HALF POPULATION STABILITY INDEX (PSI)
# =============================================================================
_section("SECTION 6: Split-Half Population Stability Index (PSI)")

_shuffled = ecl_df.sample(frac=1.0, random_state=RANDOM_SEED).reset_index(drop=True)
_half_a, _half_b = _shuffled.iloc[: len(_shuffled) // 2], _shuffled.iloc[len(_shuffled) // 2:]
_stages_present = sorted(ecl_df["ifrs9_stage"].unique())
_dist_a = _half_a["ifrs9_stage"].value_counts(normalize=True).reindex(_stages_present).fillna(0.0)
_dist_b = _half_b["ifrs9_stage"].value_counts(normalize=True).reindex(_stages_present).fillna(0.0)
_eps = 1e-6
PSI = float(sum((_dist_b[_s] - _dist_a[_s]) * np.log((_dist_b[_s] + _eps) / (_dist_a[_s] + _eps))
                 for _s in _stages_present))
_psi_verdict = "stable" if PSI < 0.10 else ("moderate shift" if PSI < 0.25 else "significant shift")
print(f"Split-half PSI on IFRS9 stage distribution: {PSI:.4f} ({_psi_verdict})")

validation_rows = [
    {"check": "Chi-square independence (stage vs default)", "result": f"p={_chi2_p:.2e}"},
    {"check": "Cram\u00e9r's V effect size", "result": f"{_cramers_v:.4f}"},
    {"check": "Two-proportion z-test (Stage 3 vs Stage 1)", "result": f"p={_z_p:.2e}"},
    {"check": "Split-half PSI", "result": f"{PSI:.4f} ({_psi_verdict})"},
]
validation_df = pd.DataFrame(validation_rows)
validation_path = PILLAR_DIRS["p3_validation_deployment"] / "p3_statistical_validation.csv"
validation_df.to_csv(validation_path, index=False)
print(f"\u2705 Saved -> {validation_path.name}")
print("\n\u2705 Section 6 complete.")


# =============================================================================
# SECTION 7: MACRO-SCENARIO SENSITIVITY ANALYSIS
# =============================================================================
_section("SECTION 7: Macro-Scenario Sensitivity Analysis")

# --- How much does total IFRS9 ECL move under EACH individual macro scenario,
#     deterministically (not probability-blended), versus the blended figure
#     Notebook 31 reported? This shows the real range decision-makers should
#     expect, not just a single point estimate. ---
DF_STAGE1 = 1.0 / ((1.0 + ANNUAL_RATE) ** STAGE1_DISCOUNT_YEARS)
DF_STAGE23 = 1.0 / ((1.0 + ANNUAL_RATE) ** STAGE23_DISCOUNT_YEARS)

_lgd_per_customer = ecl_df["severity_tier"].astype(str).map(LGD_BY_TIER).to_numpy()
_stage = ecl_df["ifrs9_stage"].to_numpy()
_pd12 = ecl_df["pd_12m"].to_numpy()
_pdlife = ecl_df["pd_lifetime"].to_numpy()

sensitivity_rows = []
for _s in MACRO_SCENARIOS:
    _pd12_s = np.clip(_pd12 * _s["pd_multiplier"], 0.0, 1.0)
    _pdlife_s = np.clip(_pdlife * _s["pd_multiplier"], 0.0, 1.0)
    _ecl_s = np.where(_stage == 1, _pd12_s * _lgd_per_customer * EAD * DF_STAGE1,
                       _pdlife_s * _lgd_per_customer * EAD * DF_STAGE23)
    sensitivity_rows.append({"scenario": _s["scenario"], "probability": _s["probability"],
                              "pd_multiplier": _s["pd_multiplier"], "total_ecl_ifrs9_usd": float(_ecl_s.sum())})
sensitivity_df = pd.DataFrame(sensitivity_rows)
_blended_total = float(ecl_df["ecl_ifrs9_usd"].sum())
print(sensitivity_df.to_string(index=False))
print(f"\nProbability-blended total (Notebook 31, real): ${_blended_total:,.0f}")
print(f"Deterministic scenario range                 : ${sensitivity_df['total_ecl_ifrs9_usd'].min():,.0f} "
      f"(best case) -- ${sensitivity_df['total_ecl_ifrs9_usd'].max():,.0f} (worst case)")

sensitivity_path = PILLAR_DIRS["p3_validation_deployment"] / "p3_macro_sensitivity.csv"
sensitivity_df.to_csv(sensitivity_path, index=False)
print(f"\u2705 Saved -> {sensitivity_path.name}")
print("\n\u2705 Section 7 complete.")


# =============================================================================
# SECTION 8: DEPLOYMENT BUNDLE -- CARRY FORWARD NOTEBOOK 30's REAL POLICY EXACTLY
# =============================================================================
_section("SECTION 8: Deployment Bundle -- Carry Forward Notebook 30's Real Policy Exactly")

# --- Every value below is read directly from ecl_policy.json -- never re-typed,
#     re-rounded, or re-derived. This is the lesson Problem 4's Notebook 28
#     learned the hard way: a second independent computation (or a rounded
#     copy) of a value the real pipeline already computed is a standing risk
#     of disagreement, however small. There is nothing to recompute here. ---
ECL_BUNDLE = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "tier_order": TIER_ORDER, "lgd_by_tier": LGD_BY_TIER, "ead_per_account_usd": EAD,
    "sicr_pd_multiple": SICR_PD_MULTIPLE, "stage3_pd_threshold": STAGE3_PD_THRESHOLD,
    "lifetime_pd_multiplier": LIFETIME_PD_MULTIPLIER, "macro_scenarios": MACRO_SCENARIOS,
    "discount_factor_stage1": DF_STAGE1, "discount_factor_stage23": DF_STAGE23,
    "portfolio_avg_pd_12m": float(ecl_df["pd_12m"].mean()),  # frozen from this real run, per Notebook 31
}
bundle_path = PILLAR_DIRS["p3_validation_deployment"] / "ecl_scoring_bundle.json"
with open(bundle_path, "w", encoding="utf-8") as f:
    json.dump(ECL_BUNDLE, f, indent=2)
print(f"\u2705 Saved -> {bundle_path.name}")
print("\n\u2705 Section 8 complete.")


# =============================================================================
# SECTION 9: GENERATE ecl_calculator.py -- STANDALONE RUNNABLE ECL FORMULA
# =============================================================================
_section("SECTION 9: Generate ecl_calculator.py -- Standalone Runnable ECL Formula")

_calculator_source = """# Standalone Expected Credit Loss (ECL) calculator for Problem 3 (IFRS9/CECL). Generated by
# Notebook 32 -- loads ecl_scoring_bundle.json (saved alongside this file) and computes IFRS9-staged
# and CECL ECL for one customer, given their real PD (from Problem 1's own deployed model -- NOT
# reproduced here) and real severity tier (from Problem 4's own deployed model). No training or
# staging-threshold-fitting happens here; every parameter is frozen from Notebook 30's real policy.
import json
from pathlib import Path

BUNDLE_PATH = Path(__file__).parent / "ecl_scoring_bundle.json"


def load_bundle(path=BUNDLE_PATH):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def compute_ecl(pd_12m, severity_tier, bundle):
    # pd_12m: float, this customer's real 12-month PD (from Problem 1's champion model).
    # severity_tier: str, one of bundle["tier_order"] (from Problem 4's real severity model).
    # Returns a dict with the IFRS9 stage, IFRS9 ECL, CECL ECL, and the intermediate PD figures --
    # matching Notebook 31's own real computation exactly (same order of operations, same rounding
    # behavior: none -- every intermediate value stays full float precision throughout).
    pd_12m = float(pd_12m)
    pd_lifetime = min(1.0, pd_12m * bundle["lifetime_pd_multiplier"])

    pd_12m_macro = 0.0
    pd_lifetime_macro = 0.0
    for s in bundle["macro_scenarios"]:
        pd_12m_macro += s["probability"] * min(1.0, pd_12m * s["pd_multiplier"])
        pd_lifetime_macro += s["probability"] * min(1.0, pd_lifetime * s["pd_multiplier"])

    is_severe = severity_tier == "Severe"
    is_elevated_tier = severity_tier in ("Moderate Severity", "Severe")
    sicr_threshold = bundle["portfolio_avg_pd_12m"] * bundle["sicr_pd_multiple"]
    if is_severe and pd_12m > bundle["stage3_pd_threshold"]:
        stage = 3
    elif (pd_12m > sicr_threshold) or is_elevated_tier:
        stage = 2
    else:
        stage = 1

    lgd = bundle["lgd_by_tier"][severity_tier]
    ead = bundle["ead_per_account_usd"]
    if stage == 1:
        ecl_ifrs9 = pd_12m_macro * lgd * ead * bundle["discount_factor_stage1"]
    else:
        ecl_ifrs9 = pd_lifetime_macro * lgd * ead * bundle["discount_factor_stage23"]
    ecl_cecl = pd_lifetime_macro * lgd * ead * bundle["discount_factor_stage23"]

    return {"ifrs9_stage": stage, "ecl_ifrs9_usd": ecl_ifrs9, "ecl_cecl_usd": ecl_cecl,
            "pd_lifetime": pd_lifetime, "pd_12m_macro_adj": pd_12m_macro,
            "pd_lifetime_macro_adj": pd_lifetime_macro}


if __name__ == "__main__":
    _bundle = load_bundle()
    print(compute_ecl(0.10, "Moderate Severity", _bundle))
"""
calculator_path = PILLAR_DIRS["p3_validation_deployment"] / "ecl_calculator.py"
with open(calculator_path, "w", encoding="utf-8") as f:
    f.write(_calculator_source)
print(f"\u2705 Saved -> {calculator_path.name}")
print("\n\u2705 Section 9 complete.")


# =============================================================================
# SECTION 10: LIVE SELF-TEST -- IMPORT ecl_calculator.py, SCORE EVERY REAL HOLDOUT CUSTOMER
# =============================================================================
_section("SECTION 10: Live Self-Test -- Import ecl_calculator.py, Score Every Real Holdout Customer")

_spec = importlib.util.spec_from_file_location("ecl_calculator", str(calculator_path))
ecl_calculator = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(ecl_calculator)

# --- Every real holdout customer is checked, not a sample -- a summation/order or edge-case bug
#     typically only surfaces for customers near a decision boundary. A tier/stage mismatch is a
#     BOUNDARY TIE (informational, not a failure) only if the underlying dollar ECL values agree to
#     within BOUNDARY_EPS -- anything larger is a real, hard-failing defect. See Problem 4's Notebook
#     28 project notes for why this distinction matters and how it was discovered. ---
BOUNDARY_EPS_USD = 1e-6
_hard_mismatches, _boundary_ties = [], []
for _rec in ecl_df.to_dict("records"):
    _res = ecl_calculator.compute_ecl(_rec["pd_12m"], str(_rec["severity_tier"]), ECL_BUNDLE)
    _stage_diff = _res["ifrs9_stage"] != int(_rec["ifrs9_stage"])
    _ecl_ifrs9_diff = abs(_res["ecl_ifrs9_usd"] - float(_rec["ecl_ifrs9_usd"]))
    _ecl_cecl_diff = abs(_res["ecl_cecl_usd"] - float(_rec["ecl_cecl_usd"]))
    if not _stage_diff and _ecl_ifrs9_diff < 1e-9 and _ecl_cecl_diff < 1e-9:
        continue
    _row = {"customer_ID": _rec["customer_ID"], "pipeline_stage": int(_rec["ifrs9_stage"]),
            "calc_stage": _res["ifrs9_stage"], "ecl_ifrs9_diff_usd": _ecl_ifrs9_diff,
            "ecl_cecl_diff_usd": _ecl_cecl_diff}
    if max(_ecl_ifrs9_diff, _ecl_cecl_diff) < BOUNDARY_EPS_USD:
        _boundary_ties.append(_row)
    else:
        _hard_mismatches.append(_row)

_n_checked = len(ecl_df)
_n_hard = len(_hard_mismatches)
_n_boundary = len(_boundary_ties)
_boundary_rate = _n_boundary / _n_checked if _n_checked else 0.0
_boundary_rate_sane = _boundary_rate <= 0.001
_self_test_passed = (_n_hard == 0) and _boundary_rate_sane

print(f"Holdout customers checked                 : {_n_checked:,}")
print(f"Hard mismatches (real defect, must be 0)   : {_n_hard:,}")
print(f"Boundary ties (ECL differs by <${BOUNDARY_EPS_USD:g}, expected float noise): {_n_boundary:,} "
      f"({_boundary_rate:.4%})")
if _hard_mismatches:
    print("\nHard mismatches (investigate):")
    print(pd.DataFrame(_hard_mismatches).head(10).to_string(index=False))
print(f"\nSelf-test {'PASSED' if _self_test_passed else 'FAILED'} -- standalone calculator "
      f"{'matches' if _self_test_passed else 'does NOT match'} the Notebook 31 pipeline on "
      f"{_n_checked - _n_hard:,}/{_n_checked:,} real holdout customers.")
print("\n\u2705 Section 10 complete.")


# =============================================================================
# SECTION 11: DEPLOYMENT READINESS CHECKLIST
# =============================================================================
_section("SECTION 11: Deployment Readiness Checklist")

_checks_passed = True


def _check(label, condition, detail="", hard=True):
    global _checks_passed
    if condition:
        print(f"\u2705 {label}")
    else:
        if hard:
            _checks_passed = False
        _mark = "\u274c" if hard else "\u26a0\ufe0f"
        print(f"{_mark} {label}  {detail}")


_check(f"Standalone calculator matches Notebook 31's own output, allowing only float-boundary ties "
       f"(hard requirement) -- {_n_hard} real mismatches, {_n_boundary} boundary ties of {_n_checked:,}",
       _self_test_passed, hard=True)
_check("Stage is statistically associated with default (chi-square p < 0.05)", _chi2_p < 0.05, hard=True)
_check("Stage 3 vs Stage 1 default-rate difference is statistically significant (z-test p < 0.05)",
       _z_p < 0.05, hard=True)
_check("Split-half PSI indicates a stable staging scheme (PSI < 0.10)", PSI < 0.10, f"(measured={PSI:.4f})",
       hard=False)

readiness_rows = [
    {"check": "Standalone calculator matches pipeline output", "result": "PASS" if _self_test_passed else "FAIL"},
    {"check": "  - real (hard) mismatches", "result": f"{_n_hard:,} of {_n_checked:,}"},
    {"check": "  - float-boundary ties (expected, not a defect)", "result": f"{_n_boundary:,} ({_boundary_rate:.4%})"},
    {"check": "Chi-square independence test", "result": f"p={_chi2_p:.2e}"},
    {"check": "Two-proportion z-test (Stage 3 vs Stage 1)", "result": f"p={_z_p:.2e}"},
    {"check": "Split-half PSI", "result": f"{PSI:.4f} ({_psi_verdict})"},
    {"check": "Cram\u00e9r's V effect size", "result": f"{_cramers_v:.4f}"},
]
readiness_df = pd.DataFrame(readiness_rows)
readiness_path = PILLAR_DIRS["p3_validation_deployment"] / "p3_deployment_readiness_checklist.csv"
readiness_df.to_csv(readiness_path, index=False)
print(readiness_df.to_string(index=False))

if not _checks_passed:
    raise RuntimeError("One or more hard deployment-readiness checks failed. See \u274c line above.")
print(f"\n\u2705 Saved -> {readiness_path.name}")
print("\n\u2705 Section 11 complete.")


# =============================================================================
# SECTION 12: INLINE CHART -- MACRO SCENARIO SENSITIVITY
# =============================================================================
_section("SECTION 12: Inline Chart -- Macro Scenario Sensitivity")

VIZ = {"ink": "#0B1F3A", "accent": "#C41E3A", "muted": "#8A93A6", "gold": "#B08D57", "surface": "#FFFFFF"}
fig, ax = plt.subplots(figsize=(7.5, 5), dpi=150)
_colors = {"Upside": VIZ["muted"], "Baseline": VIZ["gold"], "Downside": VIZ["accent"]}
ax.bar(sensitivity_df["scenario"], sensitivity_df["total_ecl_ifrs9_usd"],
       color=[_colors.get(s, VIZ["ink"]) for s in sensitivity_df["scenario"]])
ax.axhline(_blended_total, color=VIZ["ink"], linestyle="--", linewidth=1.5,
           label=f"Probability-blended (real): ${_blended_total:,.0f}")
ax.set_ylabel("Total IFRS9 ECL, USD (deterministic per scenario)")
ax.set_title("Problem 3: IFRS9 ECL Sensitivity to Macro Scenario")
ax.legend()
fig.tight_layout()
chart_path = PILLAR_DIRS["p3_validation_deployment"] / "macro_sensitivity_chart.png"
fig.savefig(chart_path, dpi=150, facecolor=VIZ["surface"])
plt.show()
plt.close(fig)
print(f"\u2705 Saved -> {chart_path.name}")
print("\n\u2705 Section 12 complete.")


# =============================================================================
# SECTION 13: WRITE NOTEBOOK 32 SUMMARY ARTIFACT
# =============================================================================
_section("SECTION 13: Write Notebook 32 Summary Artifact")

_expected_files = [validation_path, sensitivity_path, bundle_path, calculator_path, readiness_path, chart_path]
for fp in _expected_files:
    _check(f"{fp.name} exists and is non-empty", fp.exists() and fp.stat().st_size > 0)
if not _checks_passed:
    raise RuntimeError("One or more Notebook 32 output files failed to save. See \u274c line above.")

notebook_32_summary = {
    "notebook": "32_validation_deployment", "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "problem_number": 3, "problem_name": "Expected Credit Loss (IFRS9/CECL)",
    "phase": "Phase 2 -- Regulatory & Loss Provisioning",
    "chi_square_p_value": float(_chi2_p), "cramers_v": _cramers_v, "z_test_p_value": _z_p,
    "psi": PSI, "psi_verdict": _psi_verdict, "self_test_passed": bool(_self_test_passed),
    "self_test_customers_checked": _n_checked, "self_test_hard_mismatches": _n_hard,
    "self_test_boundary_ties": _n_boundary,
    "macro_sensitivity_range_usd": [float(sensitivity_df["total_ecl_ifrs9_usd"].min()),
                                     float(sensitivity_df["total_ecl_ifrs9_usd"].max())],
    "output_files": {p.name: str(p) for p in _expected_files},
}
nb32_summary_path = ARTIFACTS_DIR / "notebook_32_summary.json"
with open(nb32_summary_path, "w", encoding="utf-8") as f:
    json.dump(notebook_32_summary, f, indent=2)
print(f"\u2705 Saved -> {nb32_summary_path.name}")
print("\n\u2705 Section 13 complete.")


# =============================================================================
# SECTION 14: COMPLETION SUMMARY
# =============================================================================
_section("SECTION 14: Notebook 32 Complete -- Handoff to Notebook 33")

print("NOTEBOOK 32: STATISTICAL VALIDATION, SENSITIVITY & DEPLOYMENT -- COMPLETE")
print(f"  Chi-square p-value (stage vs default)  : {_chi2_p:.2e}")
print(f"  Two-proportion z-test p-value           : {_z_p:.2e}")
print(f"  Split-half PSI                          : {PSI:.4f} ({_psi_verdict})")
print(f"  Standalone calculator self-test          : {'PASSED' if _self_test_passed else 'FAILED'}")
print(f"  Macro sensitivity range (real)           : ${sensitivity_df['total_ecl_ifrs9_usd'].min():,.0f} -- "
      f"${sensitivity_df['total_ecl_ifrs9_usd'].max():,.0f}")
print(f"  Files produced                          : {len(_expected_files) + 1}")
for _p in _expected_files + [nb32_summary_path]:
    print(f"    - {_p.name}")
print(f"  Next notebook                            : 33_financial_impact_reporting_packaging.ipynb")
print("\n\u2705 Ready to proceed.")
